In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VelocityScaler(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.theta = nn.Parameter(torch.zeros(dim))
    def scale(self, x):
        gamma = torch.exp(self.theta)
        return x * gamma

def scaled_distance(query, context, scaler):
    rel = context - query.unsqueeze(1)
    scaled = scaler.scale(rel)
    return (scaled ** 2).sum(dim=-1)

def gather_neighbors(data, query_idx, k, scaler):
    query_coords = data['coords'][query_idx]
    context_coords = data['coords'].unsqueeze(0).expand(query_coords.shape[0], -1, -1)
    context_values = data['values'].unsqueeze(0).expand(query_coords.shape[0], -1)

    dists = scaled_distance(query_coords, context_coords, scaler)
    knn_idx = dists.topk(k, largest=False).indices

    neighbors = torch.gather(context_coords, 1, knn_idx.unsqueeze(-1).expand(-1, -1, context_coords.size(-1)))
    neighbor_vals = torch.gather(context_values, 1, knn_idx)

    rel_coords = neighbors - query_coords.unsqueeze(1)
    inputs = torch.cat([rel_coords, neighbor_vals.unsqueeze(-1)], dim=-1)

    return inputs, query_coords, data['values'][query_idx]

class FieldFormerCore(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, n_heads=4, n_layers=4, mlp_hidden=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=n_heads,
                                                   dim_feedforward=mlp_hidden,
                                                   dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_mlp = nn.Sequential(
            nn.Linear(hidden_dim, mlp_hidden),
            nn.ReLU(),
            nn.Linear(mlp_hidden, 1)
        )

    def forward(self, x):  # (B, k, D+1)
        x = self.input_proj(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.output_mlp(x).squeeze(-1)

class FieldFormerWithMemory(nn.Module):
    def __init__(self, coords, values, space_time_dim, k_neighbors=30, **model_kwargs):
        super().__init__()
        self.register_buffer('coords', torch.tensor(coords, dtype=torch.float32))
        self.register_buffer('values', torch.tensor(values, dtype=torch.float32))
        self.k = k_neighbors
        self.scaler = VelocityScaler(space_time_dim)
        self.model = FieldFormerCore(input_dim=(space_time_dim + 1), **model_kwargs)

    def forward(self, query_idx):
        data = {'coords': self.coords, 'values': self.values}
        x, query_coords, targets = gather_neighbors(data, query_idx, self.k, self.scaler)
        preds = self.model(x)
        return preds, targets, query_coords


In [3]:
import numpy as np

def compute_physics_loss(model, query_coords):
    """
    Computes the mean squared PDE residual for a batch of query coordinates.
    
    Args:
        model: FieldFormerWithMemory
        query_coords: Tensor of shape (B, 3) = (x, y, t), normalized to [0,1]
    
    Returns:
        Scalar tensor: mean squared PDE residual over batch
    """
    alpha_x = 0.01
    alpha_y = 0.001
    A = 1.0
    T = 1.0
    Nx, Ny = 64, 64
    Nt = 500
    Lx, Ly = 1.0, 1.0

    # Grid spacing (assumes uniform mesh)
    dx = Lx / (Nx - 1)
    dy = Ly / (Ny - 1)
    dt = T / (Nt - 1)

    device = query_coords.device
    B = query_coords.shape[0]

    # Define finite difference offsets for stencil (B, 7, 3)
    offsets = torch.tensor([
        [0, 0, 0],      # center
        [dx, 0, 0],     # x+
        [-dx, 0, 0],    # x-
        [0, dy, 0],     # y+
        [0, -dy, 0],    # y-
        [0, 0, dt],     # t+
        [0, 0, -dt],    # t-
    ], device=device)

    # Broadcast to (B, 7, 3)
    stencil_coords = query_coords.unsqueeze(1) + offsets.unsqueeze(0)  # (B, 7, 3)
    stencil_coords_flat = stencil_coords.view(-1, 3)  # (B*7, 3)

    # Predict u values at stencil points
    preds = []
    for i in range(stencil_coords_flat.size(0)):
        coord = stencil_coords_flat[i].unsqueeze(0)  # (1, 3)
        x_in, _, _ = gather_neighbors(
            {"coords": model.coords, "values": model.values},
            query_idx=torch.randint(0, model.coords.size(0), (1,)).to(device),
            k=model.k,
            scaler=model.scaler
        )
        model.coords[0] = coord  # override query location temporarily
        pred = model.model(x_in).squeeze()
        preds.append(pred)

    # Stack and reshape: (B, 7)
    u = torch.stack(preds).view(B, 7)
    u0, u_xp, u_xm, u_yp, u_ym, u_tp, u_tm = u.T

    # Central finite differences
    dudt = (u_tp - u_tm) / (2 * dt)
    d2udx2 = (u_xp - 2 * u0 + u_xm) / dx**2
    d2udy2 = (u_yp - 2 * u0 + u_ym) / dy**2

    # Forcing term f(x, y, t)
    x = query_coords[:, 0]
    y = query_coords[:, 1]
    t = query_coords[:, 2]
    f = A * torch.cos(np.pi * x) * torch.cos(np.pi * y) * torch.sin(4 * np.pi * t / T)

    residual = dudt - (alpha_x * d2udx2 + alpha_y * d2udy2 + f)
    return (residual ** 2).mean()


In [4]:
def train_fieldformer_from_npz(npz_path, batch_size=128, epochs=10, lr=1e-3, k_neighbors=30, device='cpu'):
    import torch
    import torch.nn as nn
    import numpy as np
    from torch.utils.data import DataLoader, Dataset
    from tqdm import tqdm

    # Load data
    data = np.load(npz_path)
    values = data['data']  # (N, T)
    coords = data['coords']  # (N, 2)

    N, T = values.shape
    coords_tiled = np.repeat(coords[None, :, :], T, axis=0).reshape(-1, 2)  # (T*N, 2)
    time_column = np.repeat(np.arange(T)[:, None], N, axis=0) / T
    coords_full = np.hstack([coords_tiled, time_column])  # (T*N, 3)
    values_flat = values.flatten()

    # Model
    model = FieldFormerWithMemory(
        coords=coords_full,
        values=values_flat,
        space_time_dim=3,
        k_neighbors=k_neighbors
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    class FlatIndexDataset(Dataset):
        def __init__(self, T, N):
            self.T = T
            self.N = N
        def __len__(self):
            return self.T * self.N
        def __getitem__(self, idx):
            return idx

    loader = DataLoader(FlatIndexDataset(T, N), batch_size=batch_size, shuffle=True)

    # Training loop
    model.train()
    for epoch in range(epochs):
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")
        total_loss = 0.0
        for batch in pbar:
            batch = batch.to(device)
            preds, targets, _ = model(batch)
            data_loss = loss_fn(preds, targets)
            physics_loss = compute_physics_loss(model, batch)
            loss = data_loss+physics_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch.size(0)
            pbar.set_postfix({"loss": total_loss / len(loader.dataset)})

    return model


In [5]:
npz_path = '/scratch/ab9738/mprnn_forward/data/synthetic/sparse_sensor_data.npz'
train_fieldformer_from_npz(npz_path, batch_size=128, epochs=10, lr=1e-3, k_neighbors=30, device='cpu')

/ext3/miniforge3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
Epoch 1/10:   0%|          | 0/391 [00:00<?, ?it/s]


RuntimeError: The size of tensor a (128) must match the size of tensor b (7) at non-singleton dimension 1